<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day07-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 7 lab, part 2: real structure comparison with Foldseek {.unnumbered}

The PyMOL parts of this lab (Sections A-E in the Day 7 lab quiz on
Canvas) are meant to be done in PyMOL's own command line, not here --
see the book page's own note that PyMOL is this course's hands-on
visualization tool. This notebook covers just the one part that's
naturally code: a real Foldseek structural comparison, backing up the
Day 7 book page's own "Comparing structures: RMSD, TM-score, and
Foldseek" section with a new real example instead of just repeating it.

Colab's stock image doesn't ship Foldseek -- installed below the same
way as any other real command-line tool.


## Install Foldseek

In [ ]:
import shutil, subprocess

if shutil.which("foldseek") is None:
    print("Installing Foldseek (not preinstalled in Colab)...")
    subprocess.run(
        ["bash", "-c",
         "wget -q https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz "
         "&& tar xzf foldseek-linux-avx2.tar.gz "
         "&& mv foldseek/bin/foldseek /usr/local/bin/"],
        check=True,
    )
else:
    print("foldseek already available, skipping install.")


## A real reference structure and three real things to compare it against

The reference: `1OJ6`, the real human neuroglobin (NGB) crystal
structure already used in the PyMOL part of this lab. Three real
structures to compare it against, spanning the same three-tier range
the book page's own hemoglobin example does (identical structure by a
different method -> related family member -> unrelated fold):

1. **The real AlphaFold DB prediction for the same protein**
   (UniProt `Q9NPG2`, neuroglobin) -- same protein, two independent
   methods (X-ray vs. AI prediction).
2. **`2HHB`** (human deoxyhemoglobin, this course's own running
   example since Day 3) -- a real, related globin-family member, not
   the same protein.
3. **`1EMA`** (green fluorescent protein) -- a real, structurally
   unrelated all-beta-barrel fold.


In [ ]:
import requests

structures = {
    "1OJ6": "https://files.rcsb.org/download/1OJ6.pdb",
    "2HHB": "https://files.rcsb.org/download/2HHB.pdb",
    "1EMA": "https://files.rcsb.org/download/1EMA.pdb",
}
for name, url in structures.items():
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    with open(f"{name}.pdb", "wb") as f:
        f.write(r.content)
    print(f"Downloaded {name}.pdb ({len(r.content)} bytes)")

# The real current AlphaFold DB model for neuroglobin (UniProt Q9NPG2) --
# fetch its real, current version rather than hardcoding a version number
# that may later be superseded.
r = requests.get("https://alphafold.ebi.ac.uk/api/prediction/Q9NPG2", timeout=15)
r.raise_for_status()
af_entry = r.json()[0]
print("Real AlphaFold model version:", af_entry["latestVersion"])
r = requests.get(af_entry["pdbUrl"], timeout=30)
r.raise_for_status()
with open("AF-Q9NPG2.pdb", "wb") as f:
    f.write(r.content)
print(f"Downloaded AF-Q9NPG2.pdb ({len(r.content)} bytes)")


## Run a real local Foldseek comparison

In [ ]:
import subprocess, os

os.makedirs("query", exist_ok=True)
os.makedirs("target", exist_ok=True)
subprocess.run(["cp", "1OJ6.pdb", "query/"], check=True)
subprocess.run(["cp", "2HHB.pdb", "1EMA.pdb", "AF-Q9NPG2.pdb", "target/"], check=True)

subprocess.run(
    ["foldseek", "easy-search", "query", "target/", "result.tsv", "tmp",
     "--format-output", "query,target,alntmscore,rmsd,fident",
     "-e", "10000"],
    check=True, capture_output=True,
)

with open("result.tsv") as f:
    rows = [line.strip().split("\t") for line in f if line.strip()]

print(f"{'query':12s} {'target':16s} {'TM-score':>9s} {'RMSD':>7s} {'fident':>7s}")
for q, t, tm, rmsd, fident in rows:
    print(f"{q:12s} {t:16s} {float(tm):>9.3f} {float(rmsd):>7.2f} {float(fident):>7.3f}")

hits_by_target_prefix = {}
for q, t, tm, rmsd, fident in rows:
    prefix = t.split(".")[0].split("_")[0]
    hits_by_target_prefix.setdefault(prefix, []).append(float(tm))

print()
for prefix, tms in hits_by_target_prefix.items():
    print(f"Best real TM-score, 1OJ6 vs {prefix}: {max(tms):.3f}  ({len(tms)} chain-pair hit(s))")
for prefix in ["2HHB", "1EMA", "AF-Q9NPG2"]:
    if prefix not in hits_by_target_prefix:
        print(f"No hits at all, 1OJ6 vs {prefix} (even at a loosened significance threshold)")


## Done

Report your own real TM-scores (they'll match the pattern above, since
this is fully deterministic real data and a real deterministic tool --
your numbers should be very close to anyone else's) on the Day 7 lab
quiz on Canvas.
